# Live slate

Set `TEAM` to an abbreviation on the latest Pinnacle board. Odds come from the newest `pinnacle_nba_*_team.json`. Starters come from the newest Rotowire file that day; a team that has already left the page keeps the last five posted earlier that day.

Layer 1 draws one game environment for that team. Layer 2 is the pregame role for each player on that team and on the opponent: his minutes, his team's form, and the opponent's form. This notebook is the live path, so those windows include every completed game before the slate. The Layer 1 draw is the script. It is not drawn again.

Layer 3 turns that role and that script into minutes. The conditional fit trains on every completed game before the slate, and each window stops before the game being scored. Simulation adds the calibration residual to the raw minute prediction, then allocates the team budget once. The live minutes are the rotation segments that realize that allocation, so the schedule does not rescale it. A segment is five players on the floor for a positive duration. Posted start, who is available, and the margin on that row choose the fives. The durations add up to the script's game minutes, so the team total is five times that length and no player plays longer than that length. A player Rotowire lists as out, or out for the season, is unavailable and gets 0 minutes. Questionable and doubtful players stay available. After the script margin is known, a posted starter is active. Everyone else is active with his close-game rate, or, when the margin is at least 20, with the chance he plays in close games or in garbage time. A script with fewer than seven active players adds the next most likely names. Inactive players get 0 minutes, and the minute budget is split across the players who are in.

Layer 4 turns those minutes and the possession counts into shots and turnovers. The model is fit on the raw logs, not on the rolling rates. The attempt pool follows the team's possessions. Both teams share one timeline, and each Layer 3 segment keeps the events with the five players on the floor. Within that five, the share follows recent attempts per minute. The live profile keeps the latest completed game, and its date has to fall after the fit. Makes and free throws then move so the players' points on that row sum to the nearest whole number of that side's scenario points.

Layer 5 credits the assists, rebounds, steals, and blocks those shots and turnovers already produced. A make can be assisted by a teammate. A miss is split between the two teams, then among the teammates who were on the floor. A steal comes from the opponent's turnovers, and a block comes from an opponent miss. The same windows decide who is credited.

Layer 6 keeps one simulation row as one box score. A prop clears when the stat is strictly above its line. The chance that two props both clear is the share of rows where both do, which is not the product of the two separate chances.

Layer 7 prices points, rebounds, assists, steals, blocks, points plus assists, points plus rebounds plus assists, and rebounds plus assists on that box score against one book. A combo line is the sum of those stats on the same row. `BOOK` starts at PrizePicks and can be set to any other saved book: underdog, novig, prophetx, pinnacle, fanduel, or draftkings. PrizePicks is quoted at -119 on both sides. For the other books the line is the two-way price whose over and under are closest. `prop_over` is the share of rows strictly above the line, and `over_ev` is the profit per dollar on that over. A push returns the stake. The five highest overs are a one-dollar-each portfolio. Its risk is the standard deviation of the combined profit on those same rows.

In [1]:
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "nba_game_simulator").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("open this notebook from the nba_quant_analysis repo")

REPO = repo_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "nba_game_simulator" / "src"))

from game_enviroment import (
    build_features,
    build_game_table,
    load_gamelogs,
    load_latest_slate,
    load_model,
    print_sim_summary,
    save_model,
    season_of,
    simulate_game,
    train,
)
from lineup import (
    attach_lineup_fill,
    build_lineup_features,
    fill_lineup_gaps,
    merge_lineup_features,
)
from role_context import attach_script, role_for_slate
from game_boxscore import Prop, assemble_box_score, format_outcome, prop_probability
from shared_events import allocate_events, fit_events, live_events, training_events
from game_opportunities import (
    allocate_game_opportunities,
    fit_opportunities,
    live_opportunities,
    training_opportunities,
)
from game_rotations import (
    allocate_rotation,
    fit_rotation,
    invalid_segment_count,
    play_rates,
    script_exposure,
    script_rates,
    segments_from_minutes,
    training_minutes,
)
from src.live.predict_points import lineups_on
from src.pipeline.silver.rotowire import normalize_team_abbreviation
from src.scrapers.arbitrage import canonical_team

TEAM = "BKN"
N_SIMS = 10000
MODEL_PATH = REPO / "nba_game_simulator" / "game_env_model.json"

In [2]:
log_paths = sorted((REPO / "data" / "silver" / "nba").glob("*/regular_season/player_gamelogs.parquet"))
if not log_paths:
    raise FileNotFoundError("no player gamelogs under data/silver/nba")
roster_path = log_paths[-1]
played = pd.read_parquet(
    roster_path,
    columns=["player_name", "player_id", "team_abbreviation", "game_date"],
)
played["game_date"] = pd.to_datetime(played["game_date"])
roster = (
    played.sort_values(["player_id", "game_date"])
    .groupby("player_id", as_index=False)
    .tail(1)[["player_name", "player_id", "team_abbreviation"]]
)

slate = load_latest_slate(TEAM, roster)
board = json.loads(Path(slate["pinnacle_file"]).read_text())["games"]
posted = lineups_on(slate["date"])

def abbr_for(label):
    wanted = canonical_team(label, "nba")
    for abbr in roster["team_abbreviation"].dropna().unique():
        normalized = normalize_team_abbreviation(abbr)
        if canonical_team(normalized, "nba") == wanted:
            return normalized
    return label

rows = []
for game in board:
    participants = game.get("participants") or []
    if len(participants) < 2:
        continue
    away_name, home_name = participants[0], participants[1]
    markets = game.get("team_markets") or {}
    total = None
    for block in markets.get("total") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "over" and line.get("points") is not None:
                total = line["points"]
                break
    home_spread = None
    for block in markets.get("spread") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "home" and line.get("points") is not None:
                home_spread = line["points"]
                break
    home_abbr = abbr_for(home_name)
    away_abbr = abbr_for(away_name)
    rows.append(
        {
            "away": away_abbr,
            "home": home_abbr,
            "home_spread": home_spread,
            "total": total,
            "away_five": " ".join(posted.get(away_abbr) or []) or "—",
            "home_five": " ".join(posted.get(home_abbr) or []) or "—",
        }
    )

print(f"Pinnacle: {Path(slate['pinnacle_file']).name}")
print(f"Rotowire: {Path(slate['lineup_file']).name}")
print(f"roster:   {roster_path.relative_to(REPO)} through {played['game_date'].max().date()}")
display(pd.DataFrame(rows))

print(
    f"{slate['team']} vs {slate['opponent']}"
    f"  home={slate['is_home']}  spread={slate['spread']}  total={slate['total']}"
)
if slate["team_lineup"] is None or slate["opponent_lineup"] is None:
    print("A five is missing from today's Rotowire files. That side uses the training-average starters.")
else:
    print(f"starter ids  {slate['team']}: {slate['team_lineup']}")
    print(f"starter ids  {slate['opponent']}: {slate['opponent_lineup']}")
for side, missing in (
    (slate["team"], slate["team_out_absent"]),
    (slate["opponent"], slate["opponent_out_absent"]),
):
    if missing:
        print(
            f"{side} listed out with no game on this roster, so no minutes:"
            f" {', '.join(missing)}"
        )

Pinnacle: pinnacle_nba_2026-11-07_143000_team.json
Rotowire: rotowire_nba_2026-11-07_143000.json
roster:   data/silver/nba/2026-27/regular_season/player_gamelogs.parquet through 2026-11-06


,away,home,home_spread,total,away_five,home_five
0,BKN,ATL,-7.0,227.0,Nolan Traore Michael Porter Chaney Johnson Jul...,Kingston Flemings Nickeil Alexander-Walker Dys...
1,HOU,PHX,6.0,224.5,Fred VanVleet Amen Thompson Kevin Durant Jabar...,Devin Booker Jalen Green Dillon Brooks Miles B...
2,NYK,CLE,5.0,223.0,Jalen Brunson Mikal Bridges Landry Shamet OG A...,James Harden Donovan Mitchell Meleek Thomas Pe...
3,ORL,GSW,-7.0,219.0,Jase Richardson Anthony Black Tristan da Silva...,Stephen Curry Brandin Podziemski Yaxel Lendebo...
4,SAS,POR,5.5,230.0,De'Aaron Fox Stephon Castle Dylan Harper Devin...,Damian Lillard Scoot Henderson Toumani Camara ...
5,WAS,OKC,-11.0,223.5,Trae Young Kyshawn George AJ Dybantsa Anthony ...,Shai Gilgeous-Alexander Ajay Mitchell Alex Car...


BKN vs ATL  home=False  spread=7.0  total=227.0
starter ids  BKN: [1642849, 1629008, 1643052, 203944, 1630549]
starter ids  ATL: [1643412, 1629638, 1630700, 1630552, 1630168]
ATL listed out with no game on this roster, so no minutes: Henri Veesaar, Keshon Gilbert, Mouhamed Gueye


In [3]:
saved = json.loads(MODEL_PATH.read_text()) if MODEL_PATH.exists() else None
if saved is not None and saved.get("version") == 3:
    model = load_model(MODEL_PATH)
    print(f"loaded {MODEL_PATH.name}  ({model.get('trained_games')} games)")
else:
    if saved is not None:
        print(
            f"{MODEL_PATH.name} is version {saved.get('version')}; "
            "retraining version 3"
        )
    raw = load_gamelogs(str(REPO / "data" / "silver" / "nba" / "*" / "regular_season" / "player_gamelogs.parquet"))
    games = build_game_table(raw)
    lineups, lineup_state = build_lineup_features(
        raw, halflife=12.0, prior_weight=4.0, season_carry=0.6, window=300,
    )
    games = merge_lineup_features(games, lineups)
    lineup_state = attach_lineup_fill(lineup_state, games)
    games = fill_lineup_gaps(games)
    games, state = build_features(
        games, halflife=12.0, prior_w=4.0, carry=0.6, window=300, min_league=30,
    )
    state["lineup"] = lineup_state
    args = argparse.Namespace(
        test_seasons=1,
        validation_season="2025-26",
        holdout_season="2026-27",
        halflife=12.0,
        prior_weight=4.0,
        season_carry=0.6,
        league_window=300,
        spread_convention="auto",
        plots=False,
    )
    model = train(games, state, args)
    save_model(model, str(MODEL_PATH))

sim = simulate_game(
    model,
    slate["team"],
    slate["opponent"],
    slate["date"],
    total=slate["total"],
    spread=slate["spread"],
    is_home=slate["is_home"],
    team_lineup=slate["team_lineup"],
    opponent_lineup=slate["opponent_lineup"],
    n=N_SIMS,
    seed=7,
)
print_sim_summary(sim, slate["total"], slate["spread"])
sim[["team_poss", "opp_poss", "pace", "team_pts", "opp_pts", "game_total", "team_margin", "n_ot"]].describe().round(2)

loaded game_env_model.json  (8367 games)

BKN at ATL | total 227.0 | team spread 7.0
               mean     sd     p05     p25     p50     p75     p95
pace         100.83   3.98   94.35   98.27  100.77  103.41  107.42
team_poss    101.42   4.86   94.38   98.39  101.02  103.99  109.64
team_pts     110.04  11.36   91.65  102.43  109.88  117.70  129.04
opp_pts      116.88  11.28   98.39  109.49  116.82  124.46  135.53
game_total   226.92  17.94  197.32  214.83  226.91  239.06  256.32
team_margin   -6.85  13.81  -29.34  -16.27   -6.89    2.35   15.69
P(OT): 5.0%


,team_poss,opp_poss,pace,team_pts,opp_pts,game_total,team_margin,n_ot
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,101.42,101.42,100.83,110.04,116.88,226.92,-6.85,0.06
std,4.86,4.85,3.98,11.36,11.28,17.94,13.81,0.26
min,82.32,82.45,82.82,60.75,75.38,158.41,-62.12,0.00
25%,98.39,98.40,98.27,102.43,109.49,214.83,-16.27,0.00
50%,101.02,101.05,100.77,109.88,116.82,226.91,-6.89,0.00
75%,103.99,103.91,103.41,117.70,124.46,239.06,2.35,0.00
max,139.84,140.84,115.21,157.23,158.74,299.29,47.69,3.00


In [4]:
ROLE_COLUMNS = [
    "season_year",
    "player_id",
    "player_name",
    "team_id",
    "team_abbreviation",
    "game_id",
    "game_date",
    "matchup",
    "min",
    "pf",
    "team_pace",
    "team_off_rating",
    "team_def_rating",
    "pace",
    "off_rating",
    "def_rating",
    "start_position",
    "pos",
    "team_pts",
    "opp_pts",
    "player_team_spread",
    "fga",
    "fgm",
    "fg3_a",
    "fg3_m",
    "fta",
    "ftm",
    "tov",
    "team_fga",
    "team_poss",
    "team_tov",
    "ast",
    "oreb",
    "dreb",
    "stl",
    "blk",
    "team_fgm",
    "team_ast",
    "team_oreb",
    "team_stl",
    "team_blk",
    "opp_dreb",
    "opp_tov",
    "opp_fga",
]
logs = pd.concat(
    [pd.read_parquet(path, columns=ROLE_COLUMNS) for path in log_paths],
    ignore_index=True,
)
logs["game_date"] = pd.to_datetime(logs["game_date"])
slate_day = pd.Timestamp(slate["date"])
history = logs.loc[logs["game_date"] < slate_day]
if history.empty:
    raise ValueError(f"no completed games before {slate['date']}")
# Who is on the team comes from this season. The windows below still use earlier seasons.
this_season = history.loc[history["season_year"].astype(str).eq(season_of(slate_day))]
latest = (
    this_season.sort_values(["player_id", "game_date", "game_id"])
    .groupby("player_id", as_index=False)
    .tail(1)
)
latest["abbr"] = latest["team_abbreviation"].map(normalize_team_abbreviation)


def team_id_for(abbr: str) -> int:
    ids = latest.loc[latest["abbr"].eq(abbr), "team_id"].dropna()
    if ids.empty:
        raise ValueError(f"no completed games for {abbr} before {slate['date']}")
    return int(ids.mode().iloc[0])


def players_for(
    abbr, own_id, other_id, is_home, spread, starter_ids, out_ids, probable_ids
):
    """Players last seen on this team, plus anyone in the posted five.

    ``out_ids`` are Rotowire out and out-for-season players. They stay on the
    roster with ``available`` 0, so the segment schedule gives them 0 minutes.
    Probable players stay available. Posted starters are active in every script.
    """
    starters = {int(player_id) for player_id in (starter_ids or [])}
    outs = {int(player_id) for player_id in (out_ids or [])}
    probable = {int(player_id) for player_id in (probable_ids or [])}
    side = latest.loc[latest["abbr"].eq(abbr)]
    posted = latest.loc[
        latest["player_id"].astype(int).isin(starters | outs | probable)
    ]
    missing = (starters | outs | probable).difference(posted["player_id"].astype(int))
    if missing:
        print(f"{abbr} posted players missing from logs: {sorted(missing)}")
    side = pd.concat([side, posted], ignore_index=True).drop_duplicates("player_id")
    if side.empty:
        raise ValueError(f"no players for {abbr} before {slate['date']}")
    on_roster = side["player_id"].astype(int)
    return pd.DataFrame(
        {
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "team": abbr,
            "game_date": slate_day,
            "team_id": own_id,
            "opp_team_id": other_id,
            "is_home": float(bool(is_home)),
            "is_starter": on_roster.isin(starters).astype(float).to_numpy(),
            "probable": on_roster.isin(probable).astype(float).to_numpy(),
            "available": (~on_roster.isin(outs)).astype(float).to_numpy(),
            "team_spread": float(spread),
        }
    )


team_abbr = normalize_team_abbreviation(slate["team"])
opp_abbr = normalize_team_abbreviation(slate["opponent"])
team_id = team_id_for(team_abbr)
opp_id = team_id_for(opp_abbr)
players = pd.concat(
    [
        players_for(
            team_abbr,
            team_id,
            opp_id,
            slate["is_home"],
            slate["spread"],
            slate["team_lineup"],
            slate["team_out"],
            slate["team_probable"],
        ),
        players_for(
            opp_abbr,
            opp_id,
            team_id,
            not slate["is_home"],
            -float(slate["spread"]),
            slate["opponent_lineup"],
            slate["opponent_out"],
            slate["opponent_probable"],
        ),
    ],
    ignore_index=True,
)
names = players[["player_id", "player_name", "team", "available"]]
role = role_for_slate(logs, players.drop(columns=["player_name", "team"]))
role = names.merge(role, on="player_id", how="left", validate="one_to_one")
role = role.merge(
    players[["player_id", "probable"]],
    on="player_id",
    how="left",
    validate="one_to_one",
)
role["side"] = role["team"].eq(team_abbr).map({True: 0, False: 1})
role = role.sort_values(
    ["side", "is_starter", "min_mean_10"],
    ascending=[True, False, False],
).drop(columns="side")
rates = play_rates(history, role)
role["play_rate"] = rates["play_rate"].to_numpy()
recent_ids = []
for side_id in role["team_id"].drop_duplicates():
    side_games = (
        history.loc[history["team_id"].eq(side_id), ["game_id", "game_date"]]
        .drop_duplicates("game_id")
        .sort_values("game_date")
        .tail(12)["game_id"]
    )
    recent_ids.extend(side_games.astype(str))
recent_ids = set(recent_ids)
pbp = pd.read_parquet(REPO / "data" / "PlayByPlay" / "s27_pbp.parquet")
pbp = pbp.loc[pbp["gameId"].astype(str).isin(recent_ids)]
exposure_logs = history.loc[
    history["game_id"].astype(str).isin(recent_ids),
    ["game_id", "player_id", "team_id", "game_date", "min", "player_name", "start_position"],
]
script = script_rates(script_exposure(pbp, exposure_logs), role)
role["close_rate"] = script["close_rate"].to_numpy()
role["blowout_rate"] = script["blowout_rate"].to_numpy()

shown = [
    "team",
    "player_name",
    "is_starter",
    "available",
    "min_mean_10",
    "play_rate",
    "close_rate",
    "blowout_rate",
    "min_std_10",
    "start_rate_10",
    "pf_mean_10",
    "season_games_prior",
    "team_pace_10",
    "team_off_10",
    "team_def_10",
    "team_rest",
    "team_b2b",
    "opp_pace_10",
    "opp_off_10",
    "opp_def_10",
    "opp_rest",
    "opp_b2b",
    "team_spread",
]
print(
    f"live role through {history['game_date'].max().date()} for {slate_day.date()}"
    f"  ({len(role)} players, latest game included)"
)
out_now = role.loc[role["available"].eq(0), ["team", "player_name"]]
if out_now.empty:
    print("Rotowire out: none on these rosters")
else:
    listed = ", ".join(
        f"{row.team} {row.player_name}" for row in out_now.itertuples(index=False)
    )
    print(f"Rotowire out: {listed}")
display(role[shown].round(2))

team_role = role.loc[role["team"].eq(team_abbr)].reset_index(drop=True)
opp_role = role.loc[role["team"].eq(opp_abbr)].reset_index(drop=True)
opp_env = sim.copy()
opp_env["team_margin"] = -opp_env["team_margin"]
opp_env["is_home"] = 1.0 - opp_env["is_home"].astype(float)
attached = {
    team_abbr: attach_script(team_role, sim),
    opp_abbr: attach_script(opp_role, opp_env),
}
for label, frame in attached.items():
    draws = frame.drop_duplicates("sim_id")
    one = frame.loc[frame["sim_id"].eq(0), "team_final_margin"]
    if one.nunique() != 1:
        raise AssertionError(f"{label} players do not share one margin")
    print(
        f"{label}: {frame['player_id'].nunique()} players share the draw"
        f"  mean margin {draws['team_final_margin'].mean():.1f}"
        f"  P(OT) {draws['went_ot'].mean():.1%}"
        f"  P(blowout) {draws['blowout'].mean():.1%}"
    )


live role through 2026-11-06 for 2026-11-07  (30 players, latest game included)
Rotowire out: BKN Egor Dëmin, ATL Jock Landale, ATL Asa Newell


,team,player_name,is_starter,available,min_mean_10,play_rate,close_rate,blowout_rate,min_std_10,start_rate_10,...,team_off_10,team_def_10,team_rest,team_b2b,opp_pace_10,opp_off_10,opp_def_10,opp_rest,opp_b2b,team_spread
0,BKN,Julius Randle,1.0,1.0,33.06,0.86,0.90,0.20,6.36,1.00,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
1,BKN,Michael Porter Jr.,1.0,1.0,32.73,0.74,0.90,0.20,7.15,1.00,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
4,BKN,Day'Ron Sharpe,1.0,1.0,22.35,0.74,0.90,0.60,4.46,0.80,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
13,BKN,Chaney Johnson,1.0,1.0,21.40,0.81,0.90,0.80,10.85,0.40,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
8,BKN,Nolan Traore,1.0,1.0,20.22,0.81,0.90,0.60,8.23,0.50,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
11,BKN,Ben Saraf,0.0,1.0,25.73,0.38,0.12,0.50,10.18,0.50,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
9,BKN,Egor Dëmin,0.0,0.0,24.45,0.45,0.50,0.20,6.06,1.00,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
12,BKN,Drake Powell,0.0,1.0,24.06,0.38,0.25,0.75,9.67,0.70,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
14,BKN,Mikel Brown Jr.,0.0,1.0,22.88,0.69,0.70,0.60,3.64,0.00,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0
5,BKN,Keon Ellis,0.0,1.0,22.20,0.86,0.90,0.40,5.98,0.30,...,106.30,116.46,3.0,0.0,103.1,114.59,111.26,1.0,1.0,7.0


BKN: 16 players share the draw  mean margin -6.8  P(OT) 5.0%  P(blowout) 19.9%
ATL: 14 players share the draw  mean margin 6.8  P(OT) 5.0%  P(blowout) 19.9%


In [5]:
trained = training_minutes(history)
rotation = fit_rotation(trained)
print(
    f"rotation fit on {len(trained):,} player-games"
    f" through {history['game_date'].max().date()}"
    "  (each window stops before that game)"
)

game_key = f"{slate_day.date()}-{team_abbr}-{opp_abbr}"
for frame in (sim, opp_env):
    frame["game_id"] = game_key
    frame["game_date"] = slate_day

rng = np.random.default_rng(7)
minutes = {
    team_abbr: allocate_rotation(
        rotation, team_role, sim, rng, method="segments"
    ),
    opp_abbr: allocate_rotation(
        rotation, opp_role, opp_env, rng, method="segments"
    ),
}
schedules = {
    team_abbr: segments_from_minutes(team_role, sim, minutes[team_abbr]),
    opp_abbr: segments_from_minutes(opp_role, opp_env, minutes[opp_abbr]),
}


def script_zero(label, side, environment, drawn):
    """One attached script, realized from that row's allocated minutes."""
    row = environment.iloc[[0]]
    game_minutes = float(row["game_minutes"].iloc[0])
    n_ot = float(row["n_ot"].iloc[0])
    margin = float(row["team_margin"].iloc[0])
    segments = schedules[label][0]
    invalid = invalid_segment_count(
        segments,
        side,
        game_minutes=game_minutes,
        n_ot=n_ot,
    )
    if invalid != 0:
        raise AssertionError(f"{label} script 0 has {invalid} invalid segments")
    total = float(drawn[0].sum())
    if abs(total - 5.0 * game_minutes) > 1e-4:
        raise AssertionError(f"{label} script 0 misses the team minute budget")
    print(
        f"{label} script 0: n_ot {int(round(n_ot))}  "
        f"team_margin {margin:+.1f}  "
        f"cap {game_minutes:.1f}  "
        f"team total {total:.1f}  "
        f"invalid segments {invalid}"
    )


script_zero(team_abbr, team_role, sim, minutes[team_abbr])
script_zero(opp_abbr, opp_role, opp_env, minutes[opp_abbr])


def minute_table(label, side, drawn):
    budget = 5.0 * sim["game_minutes"].to_numpy()
    gap = np.max(np.abs(drawn.sum(axis=1) - budget))
    if gap > 1e-4:
        raise AssertionError(f"{label} minutes miss the script budget by {gap:.4f}")
    cap = sim["game_minutes"].to_numpy()[:, None]
    if np.any(drawn > cap + 1e-6):
        raise AssertionError(f"{label} has a player above the game length")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min_mean_10": side["min_mean_10"].to_numpy(),
            "mean": drawn.mean(axis=0),
            "sd": drawn.std(axis=0),
            "p10": np.percentile(drawn, 10, axis=0),
            "p90": np.percentile(drawn, 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "mean"], ascending=[False, False])


shown_minutes = pd.concat(
    [
        minute_table(team_abbr, team_role, minutes[team_abbr]),
        minute_table(opp_abbr, opp_role, minutes[opp_abbr]),
    ],
    ignore_index=True,
)
team_means = shown_minutes.groupby("team")["mean"].sum().round(1)
print(
    f"segment minutes for {slate_day.date()}  "
    f"mean team total {team_means.to_dict()}"
)
print("sd is the spread across scripts; one script has a fixed schedule")
display(shown_minutes.round(1))

rotation fit on 179,719 player-games through 2026-11-06  (each window stops before that game)
BKN script 0: n_ot 0  team_margin -0.5  cap 48.0  team total 240.0  invalid segments 0
ATL script 0: n_ot 0  team_margin +0.5  cap 48.0  team total 240.0  invalid segments 0
segment minutes for 2026-11-07  mean team total {'ATL': 241.4, 'BKN': 241.4}
sd is the spread across scripts; one script has a fixed schedule


,team,player_name,is_starter,min_mean_10,mean,sd,p10,p90
0,BKN,Julius Randle,1.0,33.1,33.2,6.8,24.9,41.8
1,BKN,Michael Porter Jr.,1.0,32.7,32.9,6.7,24.6,41.4
2,BKN,Chaney Johnson,1.0,21.4,26.9,7.2,17.8,35.9
3,BKN,Nolan Traore,1.0,20.2,25.5,7.2,16.5,34.6
4,BKN,Day'Ron Sharpe,1.0,22.3,25.1,7.2,15.9,34.3
5,BKN,Keon Ellis,0.0,22.2,17.0,8.3,2.7,26.7
6,BKN,Noah Clowney,0.0,20.8,16.7,8.1,3.6,26.4
7,BKN,Mikel Brown Jr.,0.0,22.9,15.4,10.8,0.0,28.1
8,BKN,Moritz Wagner,0.0,16.1,14.2,7.6,0.2,23.4
9,BKN,Terance Mann,0.0,19.3,8.2,8.6,0.0,20.3


In [6]:
trained_shots = training_opportunities(history)
shot_model = fit_opportunities(history)
shot_profile = live_opportunities(
    history, players.drop(columns=["player_name", "team"])
)
shot_profile["player_id"] = shot_profile["player_id"].astype(
    players["player_id"].dtype
)
shot_profile = names.drop(columns="available").merge(
    shot_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"shot model fit on {shot_model.metadata['player_games']:,} player-games"
    f" through {shot_model.training_cutoff}"
    f"  ({len(trained_shots):,} rolling rows; each window stops before that game)"
)


def side_profile(side):
    ordered = (
        shot_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("shot profile does not match the rotation")
    return ordered


if sim["is_home"].nunique() != 1:
    raise AssertionError("the slate does not have one home flag")
slate_is_home = bool(sim["is_home"].iloc[0])
home_abbr, away_abbr = (
    (team_abbr, opp_abbr) if slate_is_home else (opp_abbr, team_abbr)
)
home_role, away_role = (
    (team_role, opp_role) if slate_is_home else (opp_role, team_role)
)
allocated = allocate_game_opportunities(
    shot_model,
    side_profile(home_role),
    side_profile(away_role),
    minutes[home_abbr],
    minutes[away_abbr],
    sim,
    rng,
    home_segments=schedules[home_abbr],
    away_segments=schedules[away_abbr],
)
shots = {home_abbr: allocated["home"], away_abbr: allocated["away"]}
for drawn in shots.values():
    if not drawn["metadata"]["segments_used"]:
        raise AssertionError("shots were not kept inside the Layer 3 segments")
for label, scenario in ((team_abbr, sim["team_pts"]), (opp_abbr, sim["opp_pts"])):
    target = np.floor(scenario.to_numpy(dtype=float) + 0.5).astype(np.int64)
    if not np.array_equal(shots[label]["PTS"].sum(axis=1), target):
        raise AssertionError(f"{label} points miss the scenario score")
for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
    unavailable = side["available"].to_numpy() == 0
    if unavailable.any() and shots[label]["FGA"][:, unavailable].sum() != 0:
        raise AssertionError(f"{label} gave a shot to an unavailable player")


def shot_table(label, side, drawn):
    if not np.array_equal(drawn["FGA"].sum(axis=1), drawn["team_fga"]):
        raise AssertionError(f"{label} attempts do not sum to the shot pool")
    if not np.array_equal(drawn["TOV"].sum(axis=1), drawn["team_tov"]):
        raise AssertionError(f"{label} turnovers do not sum to the turnover pool")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "fga_per_min": side_profile(side)["fga_per_min"].to_numpy(),
            "fga": drawn["FGA"].mean(axis=0),
            "pts": drawn["PTS"].mean(axis=0),
            "tov": drawn["TOV"].mean(axis=0),
            "pts_p10": np.percentile(drawn["PTS"], 10, axis=0),
            "pts_p90": np.percentile(drawn["PTS"], 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_shots = pd.concat(
    [
        shot_table(team_abbr, team_role, shots[team_abbr]),
        shot_table(opp_abbr, opp_role, shots[opp_abbr]),
    ],
    ignore_index=True,
)
attempt_means = {
    team_abbr: round(float(shots[team_abbr]["team_fga"].mean()), 1),
    opp_abbr: round(float(shots[opp_abbr]["team_fga"].mean()), 1),
}
print(f"live shots for {slate_day.date()}  mean team attempts {attempt_means}")
display(shown_shots.round(2))


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:38: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)
/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:82: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)
/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


shot model fit on 179,719 player-games through 2026-11-06  (179,719 rolling rows; each window stops before that game)
live shots for 2026-11-07  mean team attempts {'BKN': 86.5, 'ATL': 93.2}


,team,player_name,is_starter,fga_per_min,fga,pts,tov,pts_p10,pts_p90
0,BKN,Michael Porter Jr.,1.0,0.47,14.87,19.62,2.09,10.0,29.0
1,BKN,Julius Randle,1.0,0.43,13.87,19.32,3.20,10.0,29.0
2,BKN,Nolan Traore,1.0,0.41,10.06,12.83,1.92,5.0,21.0
3,BKN,Day'Ron Sharpe,1.0,0.31,7.51,9.90,2.05,4.0,17.0
4,BKN,Chaney Johnson,1.0,0.30,7.98,9.21,1.46,3.0,16.0
5,BKN,Mikel Brown Jr.,0.0,0.44,6.60,7.65,1.20,0.0,17.0
6,BKN,Moritz Wagner,0.0,0.37,5.12,6.90,0.78,0.0,14.0
7,BKN,Noah Clowney,0.0,0.27,4.53,5.86,0.88,0.0,12.0
8,BKN,Keon Ellis,0.0,0.27,4.43,4.91,0.66,0.0,11.0
9,BKN,Josh Minott,0.0,0.42,3.14,3.97,0.31,0.0,11.0


In [7]:
trained_events = training_events(history)
event_model = fit_events(history)
event_profile = live_events(
    history, players.drop(columns=["player_name", "team"])
)
event_profile["player_id"] = event_profile["player_id"].astype(
    players["player_id"].dtype
)
event_profile = names.drop(columns="available").merge(
    event_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"event model fit through {event_model.cutoff}"
    f"  ({len(trained_events):,} rolling rows; each window stops before that game)"
)


def event_side(side):
    ordered = (
        event_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("event profile does not match the rotation")
    return ordered


availability = {
    team_abbr: team_role["available"].to_numpy() == 1,
    opp_abbr: opp_role["available"].to_numpy() == 1,
}
events, event_details = allocate_events(
    event_model,
    {team_abbr: event_side(team_role), opp_abbr: event_side(opp_role)},
    shots,
    minutes,
    rng,
    segments=schedules,
    availability=availability,
    return_details=True,
)
if not event_details["segments_used"]:
    raise AssertionError("events were not kept inside the Layer 3 segments")
for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
    if event_details["sources"][label]["AST"] != "layer4":
        raise AssertionError(f"{label} assists were drawn again")
    if event_details["sources"][label]["OREB"] != "layer4":
        raise AssertionError(f"{label} offensive rebounds were drawn again")
    if not np.array_equal(events[label]["AST"], shots[label]["AST"]):
        raise AssertionError(f"{label} assists are not the Layer 4 assists")
    if not np.array_equal(events[label]["OREB"], shots[label]["OREB"]):
        raise AssertionError(f"{label} offensive rebounds are not the Layer 4 rebounds")
    if not np.array_equal(events[label]["REB"], events[label]["OREB"] + events[label]["DREB"]):
        raise AssertionError(f"{label} rebounds are not offensive plus defensive")
    unavailable = side["available"].to_numpy() == 0
    if unavailable.any():
        for stat in ("AST", "OREB", "DREB", "STL", "BLK"):
            if events[label][stat][:, unavailable].sum() != 0:
                raise AssertionError(f"{label} gave {stat} to an unavailable player")
for label, other in ((team_abbr, opp_abbr), (opp_abbr, team_abbr)):
    misses = shots[label]["FGA"].sum(axis=1) - shots[label]["FGM"].sum(axis=1)
    rebounds = events[label]["OREB"].sum(axis=1) + events[other]["DREB"].sum(axis=1)
    if np.any(rebounds > misses):
        raise AssertionError(f"rebounds on {label} misses exceed those misses")
    if np.any(events[other]["STL"].sum(axis=1) > shots[label]["TOV"].sum(axis=1)):
        raise AssertionError(f"steals exceed {label} turnovers")
    if np.any(events[other]["BLK"].sum(axis=1) > misses):
        raise AssertionError(f"blocks on {label} shots exceed its misses")


def event_table(label, side, drawn):
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "ast_per_min": event_side(side)["ast_per_min"].to_numpy(),
            "ast": drawn["AST"].mean(axis=0),
            "reb": drawn["REB"].mean(axis=0),
            "stl": drawn["STL"].mean(axis=0),
            "blk": drawn["BLK"].mean(axis=0),
        }
    )
    return summary.sort_values(["is_starter", "ast"], ascending=[False, False])


shown_events = pd.concat(
    [
        event_table(team_abbr, team_role, events[team_abbr]),
        event_table(opp_abbr, opp_role, events[opp_abbr]),
    ],
    ignore_index=True,
)
assist_means = {
    team_abbr: round(float(events[team_abbr]["AST"].sum(axis=1).mean()), 1),
    opp_abbr: round(float(events[opp_abbr]["AST"].sum(axis=1).mean()), 1),
}
print(f"live events for {slate_day.date()}  mean team assists {assist_means}")
display(shown_events.round(2))


event model fit through 2026-11-06  (179,719 rolling rows; each window stops before that game)
live events for 2026-11-07  mean team assists {'BKN': 23.8, 'ATL': 26.5}


,team,player_name,is_starter,ast_per_min,ast,reb,stl,blk
0,BKN,Julius Randle,1.0,0.13,4.30,6.81,0.75,0.20
1,BKN,Day'Ron Sharpe,1.0,0.12,3.05,6.63,0.96,0.94
2,BKN,Nolan Traore,1.0,0.12,2.97,3.55,1.31,0.82
3,BKN,Chaney Johnson,1.0,0.06,2.15,4.89,0.87,0.45
4,BKN,Michael Porter Jr.,1.0,0.05,1.88,7.31,1.30,0.55
5,BKN,Mikel Brown Jr.,0.0,0.15,2.14,2.15,0.46,0.19
6,BKN,Keon Ellis,0.0,0.08,1.56,1.95,0.68,0.62
7,BKN,Moritz Wagner,0.0,0.08,1.29,2.79,0.44,0.24
8,BKN,Noah Clowney,0.0,0.05,1.21,3.35,0.38,0.46
9,BKN,Terance Mann,0.0,0.09,0.80,1.24,0.31,0.12


In [8]:
box = assemble_box_score(
    sim,
    {"home": home_role["player_id"], "away": away_role["player_id"]},
    {"home": minutes[home_abbr], "away": minutes[away_abbr]},
    {"home": shots[home_abbr], "away": shots[away_abbr]},
    {"home": events[home_abbr], "away": events[away_abbr]},
    # Layer 4 already locked points with half-up rounding. Ties round differently here.
    check_score=False,
)
if box.n_sims != len(sim):
    raise AssertionError("the box score does not have one row per script")
for label, side_name in ((home_abbr, "home"), (away_abbr, "away")):
    stats = box.teams[side_name].stats
    if not np.array_equal(stats["PTS"], shots[label]["PTS"]):
        raise AssertionError(f"{label} box points are not the Layer 4 points")
    if not np.allclose(stats["MIN"], minutes[label]):
        raise AssertionError(f"{label} box minutes are not the rotation minutes")
    if not np.array_equal(stats["AST"], events[label]["AST"]):
        raise AssertionError(f"{label} box assists are not the Layer 5 assists")
    if not np.array_equal(stats["REB"], events[label]["REB"]):
        raise AssertionError(f"{label} box rebounds are not the Layer 5 rebounds")


def box_table(label, side, side_name):
    stats = box.teams[side_name].stats
    summary = pd.DataFrame(
        {
            "team": label,
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min": stats["MIN"].mean(axis=0),
            "pts": stats["PTS"].mean(axis=0),
            "fgm": stats["FGM"].mean(axis=0),
            "fga": stats["FGA"].mean(axis=0),
            "fg3m": stats["3PM"].mean(axis=0),
            "fg3a": stats["3PA"].mean(axis=0),
            "ftm": stats["FTM"].mean(axis=0),
            "fta": stats["FTA"].mean(axis=0),
            "reb": stats["REB"].mean(axis=0),
            "ast": stats["AST"].mean(axis=0),
            "stl": stats["STL"].mean(axis=0),
            "blk": stats["BLK"].mean(axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_box = pd.concat(
    [
        box_table(home_abbr, home_role, "home"),
        box_table(away_abbr, away_role, "away"),
    ],
    ignore_index=True,
)
print(f"box score for {slate_day.date()}  {box.n_sims:,} rows, one game each")
display(shown_box.drop(columns="player_id").round(2))

played = pd.DataFrame(
    format_outcome(
        box,
        0,
        stats=(
            "MIN",
            "PTS",
            "REB",
            "AST",
            "STL",
            "BLK",
            "FGM",
            "FGA",
            "3PM",
            "3PA",
            "FTM",
            "FTA",
        ),
    )
)
played = played.rename(columns={"3PM": "FG3M", "3PA": "FG3A"})
lookup = pd.concat(
    [
        home_role.assign(side="home"),
        away_role.assign(side="away"),
    ],
    ignore_index=True,
)
lookup["player"] = lookup["player_id"].map(lambda value: str(int(value)))
lookup["team"] = lookup["side"]
lookup["team_name"] = lookup["side"].map({"home": home_abbr, "away": away_abbr})
played = played.merge(
    lookup[["team", "player", "player_name", "team_name"]],
    on=["team", "player"],
    how="left",
    validate="one_to_one",
)
played["team"] = played["team_name"]
played = played.loc[
    played["MIN"] > 0,
    [
        "team",
        "player_name",
        "MIN",
        "PTS",
        "REB",
        "AST",
        "STL",
        "BLK",
        "FGM",
        "FGA",
        "FG3M",
        "FG3A",
        "FTM",
        "FTA",
    ],
]
print(f"simulation 0 for {slate_day.date()}")
display(played.reset_index(drop=True))

leaders = (
    shown_box.loc[shown_box["team"].eq(team_abbr)]
    .sort_values(["is_starter", "pts"], ascending=[False, False])
    .head(2)
)
slate_side = "home" if team_abbr == home_abbr else "away"
props = []
for row in leaders.itertuples(index=False):
    index = list(box.teams[slate_side].player_ids).index(str(int(row.player_id)))
    column = box.teams[slate_side].stats["PTS"][:, index]
    props.append(Prop(slate_side, row.player_id, "PTS", float(np.floor(np.median(column)) + 0.5)))
priced = prop_probability(box, props)
print(
    f"P({leaders.iloc[0].player_name} over {props[0].line} and "
    f"{leaders.iloc[1].player_name} over {props[1].line})"
    f" = {priced.joint:.3f}; product of the separate chances = {priced.independent_product:.3f}"
)
print(f"separate chances = {tuple(round(value, 3) for value in priced.marginals)}")


box score for 2026-11-07  10,000 rows, one game each


,team,player_name,is_starter,min,pts,fgm,fga,fg3m,fg3a,ftm,fta,reb,ast,stl,blk
0,ATL,Nickeil Alexander-Walker,1.0,33.31,21.55,7.19,16.27,3.65,9.26,3.52,4.25,4.14,2.58,1.29,0.41
1,ATL,Jalen Johnson,1.0,34.06,18.10,6.48,14.85,1.54,4.58,3.60,4.38,9.20,5.10,0.76,0.74
2,ATL,Onyeka Okongwu,1.0,32.15,14.49,5.49,11.34,1.64,4.69,1.87,2.35,8.58,2.69,0.93,0.85
3,ATL,Dyson Daniels,1.0,32.72,12.33,5.05,10.50,0.71,2.41,1.52,2.10,7.05,4.51,1.97,0.53
4,ATL,Kingston Flemings,1.0,28.55,11.65,4.60,9.33,1.23,3.07,1.22,1.58,4.20,4.29,0.79,0.50
5,ATL,Aaron Wiggins,0.0,23.93,12.34,4.71,9.96,2.06,5.19,0.86,1.14,3.81,1.88,0.76,0.52
6,ATL,CJ McCollum,0.0,19.23,9.60,3.50,7.96,1.26,3.43,1.35,1.68,2.62,2.25,0.60,0.34
7,ATL,Luguentz Dort,0.0,21.69,9.24,3.20,7.39,1.46,3.83,1.39,1.75,3.19,1.63,0.63,0.31
8,ATL,Corey Kispert,0.0,4.70,2.64,0.91,1.78,0.48,1.03,0.34,0.45,0.60,0.38,0.07,0.11
9,ATL,Zuby Ejiofor,0.0,4.33,2.00,0.75,1.50,0.23,0.60,0.28,0.36,0.94,0.48,0.15,0.07


simulation 0 for 2026-11-07


,team,player_name,MIN,PTS,REB,AST,STL,BLK,FGM,FGA,FG3M,FG3A,FTM,FTA
0,ATL,Jalen Johnson,28.339813,17,13,2,1,2,5,12,1,4,6,7
1,ATL,Nickeil Alexander-Walker,36.461268,12,9,5,0,1,3,16,1,7,5,6
2,ATL,Dyson Daniels,32.848467,4,8,2,1,1,1,5,0,0,2,4
3,ATL,Onyeka Okongwu,39.619494,15,17,3,0,3,6,15,3,7,0,0
4,ATL,Kingston Flemings,32.688402,16,5,6,1,2,5,10,3,5,3,4
5,ATL,Aaron Wiggins,28.033397,26,10,3,2,1,10,15,4,7,2,3
6,ATL,Luguentz Dort,26.985049,17,6,2,0,0,6,12,2,3,3,4
7,ATL,RayJ Dennis,15.024110,9,2,1,1,1,3,12,2,9,1,2
8,BKN,Julius Randle,27.236480,13,3,4,2,0,4,14,1,4,4,5
9,BKN,Michael Porter Jr.,37.957276,27,4,1,1,0,9,27,4,13,5,6


P(Michael Porter Jr. over 19.5 and Julius Randle over 19.5) = 0.241; product of the separate chances = 0.229
separate chances = (0.484, 0.474)


In [9]:
BOOK = "novig"

from game_boxscore import portfolio_risk, price_over, stat_values
from src.live.prop_markets import attach_roster, book_names, latest_prop_markets

markets, market_audit = latest_prop_markets(REPO / "data" / "props", BOOK)
others = ", ".join(name for name in book_names() if name != market_audit["book"])
print(
    f"Layer 7 {market_audit['book']} from {market_audit['source_file']}. "
    f"Change BOOK to {others}."
)
if market_audit["book"] == "prizepicks":
    print("PrizePicks has no two-way price. Both sides are -119.")

roster = pd.concat(
    [
        home_role.assign(team=home_abbr, side="home"),
        away_role.assign(team=away_abbr, side="away"),
    ],
    ignore_index=True,
)
offered = attach_roster(
    markets,
    roster.loc[:, ["team", "side", "player_id", "player_name"]],
)
slate_dates = sorted(set(pd.to_datetime(offered["slate_date"]).dt.date))
if offered.empty:
    print(f"no {market_audit['book']} lines matched this game")
elif slate_dates != [slate_day.date()]:
    print(f"market dates {slate_dates}; the slate date is {slate_day.date()}")


def stat_board(stat, label):
    rows = []
    ranked = []
    for offer in offered.loc[offered["stat"].eq(stat)].itertuples(index=False):
        index = list(box.teams[offer.side].player_ids).index(offer.player_id)
        minutes = box.teams[offer.side].stats["MIN"][:, index]
        column = stat_values(box, offer.side, stat)[:, index]
        priced = price_over(
            box,
            Prop(offer.side, offer.player_id, stat, float(offer.line)),
            float(offer.over_american),
        )
        rows.append(
            {
                "team": offer.team,
                "player_name": offer.player_name,
                "min": float(minutes.mean()),
                label: float(column.mean()),
                f"{label}_line": float(offer.line),
                "odds": int(offer.over_american),
                "prop_over": priced.p_over,
                "over_ev": priced.ev,
            }
        )
        ranked.append(
            {
                "team": offer.team,
                "player_name": offer.player_name,
                "stat": label,
                "min": float(minutes.mean()),
                "line": float(offer.line),
                "odds": int(offer.over_american),
                "prop_over": priced.p_over,
                "over_ev": priced.ev,
                "side": offer.side,
                "player_id": offer.player_id,
                "box_stat": stat,
            }
        )
    board = pd.DataFrame(rows)
    if not board.empty:
        board = board.sort_values("over_ev", ascending=False)
    return board, pd.DataFrame(ranked)


ranked_rows = []
for stat, label in (
    ("PTS", "pts"),
    ("REB", "reb"),
    ("AST", "ast"),
    ("STL", "stl"),
    ("BLK", "blk"),
    ("PA", "p+a"),
    ("PRA", "p+r+a"),
    ("RA", "r+a"),
):
    board, ranked = stat_board(stat, label)
    if not ranked.empty:
        ranked_rows.append(ranked)
    print(f"{label} overs for {slate_day.date()}")
    if board.empty:
        print(f"no {label} line for this game")
        continue
    shown = board.copy()
    shown["min"] = shown["min"].round(2)
    shown[label] = shown[label].round(2)
    shown["prop_over"] = shown["prop_over"].round(3)
    shown["over_ev"] = shown["over_ev"].round(3)
    display(shown.reset_index(drop=True))

if not ranked_rows:
    print("no priced props for this game")
else:
    ranked = pd.concat(ranked_rows, ignore_index=True)
    top = ranked.sort_values("over_ev", ascending=False).head(5).reset_index(drop=True)
    print(f"top {len(top)} overs by expected value")
    shown_top = top.drop(columns=["side", "player_id", "box_stat"]).copy()
    shown_top["min"] = shown_top["min"].round(2)
    shown_top["prop_over"] = shown_top["prop_over"].round(3)
    shown_top["over_ev"] = shown_top["over_ev"].round(3)
    display(shown_top)
    legs = [
        Prop(row.side, row.player_id, row.box_stat, float(row.line))
        for row in top.itertuples(index=False)
    ]
    held = portfolio_risk(box, legs, top["odds"].tolist())
    print(
        f"portfolio of these {held.n}, $1 on each over. "
        f"expected profit = {held.expected_profit:.3f}; "
        f"risk = {held.risk:.3f}; "
        f"P(profit < 0) = {held.p_loss:.3f}; "
        f"P(all clear) = {held.joint:.3f}; "
        f"product of the separate chances = {held.independent_product:.3f}"
    )


Layer 7 novig from novig_nba_2026-11-07_144501_props.json. Change BOOK to prizepicks, underdog, prophetx, pinnacle, fanduel, draftkings.
pts overs for 2026-11-07


,team,player_name,min,pts,pts_line,odds,prop_over,over_ev
0,ATL,Kingston Flemings,28.55,11.65,8.5,-120,0.686,0.258
1,ATL,Dyson Daniels,32.72,12.33,10.5,-102,0.606,0.200
2,BKN,Nolan Traore,25.51,12.83,11.5,111,0.549,0.158
3,ATL,Nickeil Alexander-Walker,33.31,21.55,19.5,-106,0.589,0.144
4,BKN,Day'Ron Sharpe,25.08,9.90,8.5,-102,0.568,0.125
5,ATL,Onyeka Okongwu,32.15,14.49,12.5,-117,0.601,0.115
6,ATL,Luguentz Dort,21.69,9.24,8.5,104,0.536,0.094
7,BKN,Julius Randle,33.23,19.32,20.5,-100,0.416,-0.167
8,ATL,Jalen Johnson,34.06,18.10,19.5,-125,0.397,-0.286
9,BKN,Michael Porter Jr.,32.88,19.62,22.5,104,0.335,-0.317


reb overs for 2026-11-07


,team,player_name,min,reb,reb_line,odds,prop_over,over_ev
0,ATL,Kingston Flemings,28.55,4.20,2.5,-133,0.711,0.245
1,ATL,Nickeil Alexander-Walker,33.31,4.14,2.5,-147,0.723,0.215
2,BKN,Nolan Traore,25.51,3.55,1.5,-203,0.790,0.180
3,ATL,Onyeka Okongwu,32.15,8.58,7.5,-108,0.559,0.076
4,ATL,Dyson Daniels,32.72,7.05,6.5,-106,0.522,0.015
5,ATL,CJ McCollum,19.23,2.62,2.5,111,0.466,-0.018
6,BKN,Michael Porter Jr.,32.88,7.31,7.5,121,0.432,-0.044
7,ATL,Luguentz Dort,21.69,3.19,2.5,-153,0.556,-0.080
8,ATL,Jalen Johnson,34.06,9.20,9.5,108,0.430,-0.106
9,BKN,Julius Randle,33.23,6.81,6.5,-121,0.489,-0.107


ast overs for 2026-11-07


,team,player_name,min,ast,ast_line,odds,prop_over,over_ev
0,ATL,Luguentz Dort,21.69,1.63,1.5,153,0.479,0.211
1,BKN,Day'Ron Sharpe,25.08,3.05,2.5,-115,0.565,0.056
2,ATL,Onyeka Okongwu,32.15,2.69,2.5,106,0.498,0.025
3,ATL,Kingston Flemings,28.55,4.29,4.5,104,0.425,-0.133
4,BKN,Julius Randle,33.23,4.30,4.5,-102,0.430,-0.149
5,ATL,Nickeil Alexander-Walker,33.31,2.58,2.5,-150,0.485,-0.192
6,ATL,Dyson Daniels,32.72,4.51,4.5,-150,0.469,-0.218
7,BKN,Michael Porter Jr.,32.88,1.88,2.5,153,0.292,-0.261
8,BKN,Nolan Traore,25.51,2.97,3.5,-141,0.352,-0.397
9,ATL,CJ McCollum,19.23,2.25,3.5,122,0.225,-0.501


... (9 outputs left out)

,team,player_name,min,r+a,r+a_line,odds,prop_over,over_ev
0,ATL,Onyeka Okongwu,32.15,11.27,10.5,107,0.527,0.092
1,ATL,Kingston Flemings,28.55,8.49,11.5,413,0.203,0.043
2,ATL,Dyson Daniels,32.72,11.56,11.5,-117,0.483,-0.104
3,BKN,Julius Randle,33.23,11.11,11.5,-102,0.433,-0.142
4,BKN,Day'Ron Sharpe,25.08,9.67,10.5,-126,0.395,-0.291
5,ATL,Jalen Johnson,34.06,14.30,16.5,-102,0.313,-0.379


top 5 overs by expected value


,team,player_name,stat,min,line,odds,prop_over,over_ev
0,ATL,Kingston Flemings,p+r+a,28.55,16.5,-111,0.672,0.277
1,ATL,Kingston Flemings,pts,28.55,8.5,-120,0.686,0.258
2,ATL,Kingston Flemings,reb,28.55,2.5,-133,0.711,0.245
3,ATL,Nickeil Alexander-Walker,reb,33.31,2.5,-147,0.723,0.215
4,ATL,Luguentz Dort,ast,21.69,1.5,153,0.479,0.211


portfolio of these 5, $1 on each over. expected profit = 1.206; risk = 2.455; P(profit < 0) = 0.275; P(all clear) = 0.170; product of the separate chances = 0.113


In [10]:
from itertools import combinations

from game_boxscore import Prop, prop_probability

positive = ranked.loc[ranked["over_ev"] > 0].reset_index(drop=True)
pairs = []
for i, j in combinations(range(len(positive)), 2):
    a = positive.iloc[i]
    b = positive.iloc[j]
    priced = prop_probability(
        box,
        [
            Prop(a.side, a.player_id, a.box_stat, float(a.line)),
            Prop(b.side, b.player_id, b.box_stat, float(b.line)),
        ],
    )
    if a.player_id == b.player_id:
        link = "same player"
    elif a.team == b.team:
        link = "teammates"
    else:
        link = "opponents"
    pairs.append(
        {
            "link": link,
            "a": f"{a.player_name} {a.stat} {a.line:g}",
            "b": f"{b.player_name} {b.stat} {b.line:g}",
            "p_a": priced.marginals[0],
            "p_b": priced.marginals[1],
            "joint": priced.joint,
            "product": priced.independent_product,
            "gap": priced.joint - priced.independent_product,
        }
    )

board = pd.DataFrame(pairs).sort_values("gap", ascending=False).reset_index(drop=True)
print(
    f"correlation pairs among {len(positive)} overs with over_ev > 0, "
    f"{market_audit['book']}, {slate_day.date()}, {box.n_sims:,} rows"
)
shown = board.copy()
for column in ("p_a", "p_b", "joint", "product", "gap"):
    shown[column] = shown[column].round(3)
display(shown)
print(
    "top row is the parlay: they clear together more often than the product. "
    "bottom row is the pair of singles."
)

correlation pairs among 27 overs with over_ev > 0, novig, 2026-11-07, 10,000 rows


,link,a,b,p_a,p_b,joint,product,gap
0,same player,Nickeil Alexander-Walker pts 19.5,Nickeil Alexander-Walker p+a 23.5,0.589,0.518,0.508,0.305,0.203
1,same player,Onyeka Okongwu pts 12.5,Onyeka Okongwu p+a 15.5,0.601,0.577,0.544,0.347,0.198
2,same player,Nickeil Alexander-Walker p+a 23.5,Nickeil Alexander-Walker p+r+a 25.5,0.518,0.622,0.510,0.322,0.187
3,same player,Onyeka Okongwu reb 7.5,Onyeka Okongwu r+a 10.5,0.559,0.527,0.482,0.295,0.187
4,same player,Luguentz Dort pts 8.5,Luguentz Dort p+r+a 13.5,0.536,0.545,0.479,0.292,0.187
...,...,...,...,...,...,...,...,...
346,teammates,Luguentz Dort pts 8.5,Onyeka Okongwu reb 7.5,0.536,0.559,0.289,0.300,-0.011
347,teammates,Luguentz Dort pts 8.5,Onyeka Okongwu p+r+a 22.5,0.536,0.633,0.328,0.339,-0.011
348,teammates,Nickeil Alexander-Walker pts 19.5,Luguentz Dort p+r+a 13.5,0.589,0.545,0.309,0.321,-0.012
349,teammates,Nickeil Alexander-Walker p+a 23.5,Luguentz Dort p+r+a 13.5,0.518,0.545,0.270,0.282,-0.013


top row is the parlay: they clear together more often than the product. bottom row is the pair of singles.
